# Event charts with anytimeline

An event chart draws one mark per event at the moment it happened, without a duration, so
the **order** of what happened is easy to see. Click a plot, then scroll to zoom time; drag
to pan. The toolbar has **Pan**, **Zoom** (drag a time range) and **Select** (drag a box or
click an event); `?` lists every gesture and key, `f` shows the plot full screen. `t` (or
Time | Order) changes the scale.

In [ ]:
import numpy as np

from anytimeline import Route, Timeline
from anytimeline.synthetic import instants, service_log

log = service_log()  # a dict of columns; Polars, pandas, PyArrow and DuckDB tables work too
len(log["time"])

## Events on lanes

One lane per service; colour by level, symbol by event. Events that would overlap are spread over sub-rows.

In [ ]:
services = ["gateway", "auth", "api", "cache", "db"]
plot = Timeline(title="Requests through services", lane_order=services)
plot.events(
    log,
    time="time",
    lane="service",
    color="level",
    symbol="event",
    order="seq",
    palette={"info": "#4c78a8", "warning": "#f58518", "error": "#e45756"},
    tooltip=["request", "user", "bytes"],
)
plot

## The order of events

`xscale="sequence"` spaces events evenly by rank: a burst within a millisecond reads as clearly as events minutes apart. A `route` joins each request's events in order.

In [ ]:
ordered = Timeline(title="Requests in order", lane_order=services, xscale="sequence")
ordered.events(
    log,
    time="time",
    lane="service",
    color="request",
    order="seq",
    route=Route("request"),
    text="{event}",
    tooltip=["request"],
)
middle = log["time"][len(log["time"]) // 2]
ordered.zoom_to(middle, middle + np.timedelta64(20, "ms"))
ordered

## Selecting events

Choose **Select** in the toolbar (or press `s`), then click an event or drag a box; Shift adds, Alt removes. With Pan, Shift-click adds an event. The selection is rows of your table:

In [ ]:
rows = plot.selection.get("events", [])
{k: np.asarray(v)[rows] for k, v in log.items() if k in ("time", "service", "event", "request")}

## A million events

All drawn on the GPU (WebGPU or WebGL2; Canvas 2D on machines without a usable GPU).

In [ ]:
big = instants(1_000_000, lanes=40)
Timeline(title="1,000,000 events").events(big, lane="lane", color="kind", size=5)

## From nanoseconds to centuries

The time axis adapts to the zoom; dates before 1677 work too.

In [ ]:
history = {
    "time": np.array(
        [
            "1066-10-14",
            "1215-06-15",
            "1492-10-12",
            "1789-07-14",
            "1914-06-28",
            "1969-07-20",
            "1989-11-09",
        ],
        dtype="datetime64[D]",
    ),
    "region": ["Europe", "Europe", "Americas", "Europe", "Europe", "Space", "Europe"],
    "event": [
        "Hastings",
        "Magna Carta",
        "Columbus lands",
        "Bastille",
        "Sarajevo",
        "Moon landing",
        "Berlin Wall",
    ],
}
Timeline(title="Centuries").events(history, lane="region", text="event", symbol="diamond", size=10)

In [ ]:
rng = np.random.default_rng(0)
ns = np.sort(rng.integers(0, 50_000, 300)).astype("timedelta64[ns]")
Timeline(title="Nanoseconds").events(
    {"time": ns, "cpu": rng.integers(0, 4, 300)}, lane="cpu", symbol="tick", size=12
)

## Order without a clock

Events with only an order (steps of a protocol, logical clocks):

In [ ]:
steps = {
    "step": [1, 2, 3, 4, 5, 6],
    "actor": ["client", "server", "db", "server", "client", "server"],
    "what": ["request", "parse", "query", "render", "receive", "log"],
}
Timeline(title="Order only").events(
    steps, time=None, order="step", lane="actor", text="what", route=Route("actor", color="theme")
)